# Week 3 — Member 4: Indexes and Reading EXPLAIN Output

**Your learning outcome:** Be able to read a PostgreSQL `EXPLAIN` plan and explain why an index helps or hurts.

This notebook builds on the SQL you have already practised: `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT`, `COUNT(*)`, `AND`, `OR`, and `IS NULL`.

## How to use this notebook

1. Read each explanation before running the code.
2. Run code cells in order in a PostgreSQL environment such as your PgQuest SQL editor.
3. Look at the output, not just whether the query succeeds.
4. Try the short questions before reading the answers.

> **Important:** The SQL cells are for PostgreSQL. If your editor starts a new database session, a temporary table from an earlier session may no longer exist. This notebook creates its own practice table so you can start cleanly.

## 1. Quick recap: what are we working with?

A **database** is an organised collection of information. A **table** stores related records in rows and columns.

- A **row** is one record — for example, one employee.
- A **column** is one type of information — for example, `salary`.
- A **query** is a request we send to the database.
- PostgreSQL is the database management system that stores and works with the data.

We will use a small employee table. Each employee has an ID, a name, and a salary.

In [ ]:
SELECT version();

`SELECT` asks PostgreSQL to return information. `version()` is a PostgreSQL function that reports which PostgreSQL version is running.

## 2. Create the practice table

The next cell creates a temporary table with 10,000 employees.

New terms:

- `DROP TABLE IF EXISTS`: remove the practice table if it already exists. `IF EXISTS` prevents an error when the table is not there.
- `CREATE TEMP TABLE`: create a temporary table. A temporary table is normally available only during the current database session.
- `AS`: tells PostgreSQL to use the result of the query that follows to fill the table.
- `generate_series(1, 10000)`: a PostgreSQL function that generates the numbers 1 through 10,000.
- `||`: joins text together. For example, `'Employee ' || 7` becomes `Employee 7`.

This cell resets only the practice table named `employees_demo`. Do not use this pattern on a real table containing important data.

In [ ]:
DROP TABLE IF EXISTS employees_demo;

CREATE TEMP TABLE employees_demo AS
SELECT
    g AS id,
    'Employee ' || g AS name,
    g * 100 AS salary
FROM generate_series(1, 10000) AS g;

Check how many records were created. `COUNT(*)` counts all rows. The `*` means all rows, not all columns in this context.

In [ ]:
SELECT COUNT(*)
FROM employees_demo;

You should get **10,000**.

Look at a few rows. `LIMIT 5` tells PostgreSQL to return no more than five rows.

In [ ]:
SELECT *
FROM employees_demo
LIMIT 5;

## 3. What is an index?

An **index** is an additional data structure PostgreSQL can use to find rows more efficiently.

Think of the index at the back of a textbook. Instead of reading every page to find a topic, you look up the topic in the index and go to the relevant page.

A database index works in a similar way: it stores indexed values and references to where the corresponding table rows can be found. The original table remains the place where the employee records are stored.

PostgreSQL commonly uses a **B-tree** index by default. A B-tree is a balanced, tree-shaped search structure that keeps values organised so PostgreSQL can navigate to a value or range without checking every table row.

### Create an index on salary

New SQL terms:

- `CREATE INDEX`: create an index.
- `idx_employees_demo_salary`: the name we give this index. The name is our choice.
- `ON`: identifies the table the index belongs to.
- `employees_demo(salary)`: says to index the `salary` column in the `employees_demo` table.

In [ ]:
CREATE INDEX idx_employees_demo_salary
ON employees_demo(salary);

The index does not add a new employee or change anyone's salary. It creates an additional search structure for the `salary` column.

## 4. Read an EXPLAIN plan before and after indexing

`EXPLAIN` asks PostgreSQL to show the plan it intends to use for a query. It does **not** run the `SELECT` query itself.

First, remove the index so we can see the plan without it. This is safe for our practice table.

In [ ]:
DROP INDEX IF EXISTS idx_employees_demo_salary;

EXPLAIN
SELECT *
FROM employees_demo
WHERE salary = 500000;

### Understand the first plan

You may see something like:

```text
Seq Scan on employees_demo
  Filter: (salary = 500000)
```

The exact cost and row estimates can differ between environments.

- **`Seq Scan` (Sequential Scan):** PostgreSQL checks the table rows in sequence to find rows that match.
- **`Filter`:** the condition PostgreSQL checks on the rows.
- **`salary = 500000`:** the condition from our `WHERE` clause.

A sequential scan is not automatically bad. It can be a sensible choice for a small table or when a query needs a large proportion of the rows.

### Create the index again and compare

In [ ]:
CREATE INDEX idx_employees_demo_salary
ON employees_demo(salary);

EXPLAIN
SELECT *
FROM employees_demo
WHERE salary = 500000;

You may now see an **Index Scan** plan, for example:

```text
Index Scan using idx_employees_demo_salary on employees_demo
  Index Cond: (salary = 500000)
```

- **`Index Scan`:** PostgreSQL uses an index to locate matching row(s), then fetches the requested table data.
- **`using idx_employees_demo_salary`:** identifies the index used.
- **`Index Cond` (Index Condition):** the condition used to search the index.

The estimated cost is often lower for this selective search after indexing. But do not memorise a particular cost number: costs are estimates, not milliseconds or money.

## 5. What do the numbers in EXPLAIN mean?

A plan node may look like this:

```text
Index Scan using idx_employees_demo_salary on employees_demo
(cost=0.29..8.30 rows=1 width=21)
```

- **`cost=0.29..8.30`:** the planner's estimated startup cost and total cost, in PostgreSQL's internal cost units. These are not seconds.
- **`rows=1`:** the estimated number of rows this plan node will return.
- **`width=21`:** the estimated average size, in bytes, of each returned row.

The planner estimates these values using information it has about the table. The estimate may not exactly match what happens when the query runs.

## 6. EXPLAIN ANALYZE: compare estimates with what happened

`EXPLAIN ANALYZE` both displays the plan **and executes the query**. It adds measured information such as actual rows and execution time.

Use it with a `SELECT` query like this one. Be more careful with statements that change data: `EXPLAIN ANALYZE` really executes the statement.

In [ ]:
EXPLAIN ANALYZE
SELECT *
FROM employees_demo
WHERE salary = 500000;

### New terms you may see

- **`actual time=a..b`:** measured time in milliseconds. The first number is the time until the node produces its first row; the second is the time until it finishes.
- **`actual rows`:** how many rows the node actually produced.
- **`loops`:** how many times that plan node ran.
- **`Planning Time`:** time spent working out the plan.
- **`Execution Time`:** time spent executing the query.
- **`Rows Removed by Filter`:** rows examined by a filter but rejected because they did not match.

Compare `rows` (estimated) with `actual rows` (measured). A large difference can mean the planner's information or estimate was inaccurate.

### Refresh planner statistics with ANALYZE

`ANALYZE employees_demo;` asks PostgreSQL to collect/update statistics about the table. The planner uses statistics to estimate how many rows a condition may return.

`ANALYZE` is different from `EXPLAIN ANALYZE`:
- `ANALYZE employees_demo;` updates table statistics.
- `EXPLAIN ANALYZE SELECT ...;` executes a query and reports its plan and measurements.

In [ ]:
ANALYZE employees_demo;

EXPLAIN
SELECT *
FROM employees_demo
WHERE salary = 500000;

## 7. The main scan types

The scan type tells you how PostgreSQL is accessing the table or index.

### A. Sequential Scan (`Seq Scan`)
Checks table rows in sequence. It can be efficient when the table is small or many rows are needed.

### B. Index Scan
Uses an index to locate matching row references, then visits the table to retrieve requested columns.

### C. Bitmap Index Scan + Bitmap Heap Scan
These often appear as a pair:
- **Bitmap Index Scan:** searches the index and collects locations of matching rows.
- **Bitmap Heap Scan:** uses those locations to fetch rows from the table. Grouping locations can make fetching several matching rows more efficient.

A bitmap plan is not simply “better” than an Index Scan. PostgreSQL chooses based on its cost estimates and the query.

### D. Index Only Scan
Can return the requested columns directly from the index when the index contains all the columns the query needs. PostgreSQL may still visit the table to check row visibility, so `Heap Fetches` can be greater than zero.

## 8. Try an Index Only Scan

Our salary index contains the `salary` value. This query requests only `salary`, so the index may be able to provide the result without retrieving other employee columns from the table.

In [ ]:
EXPLAIN ANALYZE
SELECT salary
FROM employees_demo
WHERE salary = 500000;

Now request every column with `SELECT *`. The salary index does not contain every employee column, so PostgreSQL generally needs to visit the table to retrieve the remaining values.

In [ ]:
EXPLAIN ANALYZE
SELECT *
FROM employees_demo
WHERE salary = 500000;

## 9. Selectivity: how many rows match?

**Selectivity** describes how narrowly a condition identifies rows.

- A highly selective condition matches a small proportion of the table. Example: a unique email address.
- A low-selectivity condition matches a large proportion. Example: a query that returns most employees.

Imagine a table with 100,000 employees:
- Searching for one unique email may return one row. An index is likely to help.
- Searching for a gender value that matches 90,000 employees may return most of the table. Reading the table sequentially may be cheaper than using the index and fetching a large number of rows.

The number of rows returned is only one factor. Table size, data distribution, cached pages, and the columns requested also affect the planner's decision.

## 10. Why not index every column?

Indexes have benefits and costs.

**Benefits**
- Can make selective searches faster.
- Can help with some range conditions such as `salary > 750000`.
- Can help some `ORDER BY` operations.

**Costs**
- Use extra storage.
- Need maintenance when rows are inserted, updated, or deleted.
- Too many indexes can make writes slower.
- An index may not help a query that returns a large proportion of the table.

**Rule of thumb:** consider an index for columns that are searched often and where the search narrows the result set substantially. Then test with `EXPLAIN` / `EXPLAIN ANALYZE`; do not assume an index will always be used.

## 11. Composite (multicolumn) indexes and column order

A **composite index** (also called a multicolumn index) stores more than one column in one index.

Example:

In [ ]:
CREATE INDEX idx_employees_demo_name_salary
ON employees_demo(name, salary);

This index is ordered by `name` first, and then by `salary` within matching names. For a B-tree index, the leading (leftmost) column is especially important for navigating the index.

So an index on `(name, salary)` is naturally useful for queries filtering by `name`, or by `name` and `salary` together. It is generally not an effective way to search by `salary` alone.

Try comparing these plans:

In [ ]:
EXPLAIN
SELECT name, salary
FROM employees_demo
WHERE name = 'Employee 5000'
  AND salary = 500000;

In [ ]:
EXPLAIN
SELECT name, salary
FROM employees_demo
WHERE salary = 500000;

Do not expect every environment to choose the same plan. PostgreSQL makes the decision using its estimates. The key lesson is that column order matters in a multicolumn B-tree index.

## 12. Practical checklist for reading a plan

When you see an `EXPLAIN` output, ask:

1. **What scan type is used?** `Seq Scan`, `Index Scan`, `Bitmap Heap Scan`, or `Index Only Scan`?
2. **Is an index named?** Which one?
3. **Where is the condition applied?** Look for `Filter`, `Index Cond`, or `Recheck Cond`.
4. **How many rows are estimated?** Look for `rows=`.
5. **If using EXPLAIN ANALYZE, how many rows actually appeared?** Look for `actual rows=`.
6. **Are many rows being removed by a filter?** That can indicate many rows were checked but did not match.
7. **Does the index actually suit the query?** Consider selectivity, requested columns, and multicolumn index order.
8. **What are the trade-offs?** Indexes use storage and add work to inserts, updates, and deletes.

## 13. Mini practice — answer before checking

**Question 1:** If PostgreSQL reports `Seq Scan`, what is it doing?

**Answer:** It is scanning the table rows in sequence and checking which rows match.

**Question 2:** Does `EXPLAIN` execute the `SELECT` query?

**Answer:** No. It shows the planned approach. `EXPLAIN ANALYZE` executes the query and reports measured results.

**Question 3:** Does creating an index guarantee PostgreSQL will use it?

**Answer:** No. PostgreSQL chooses the plan it estimates will be cheapest.

**Question 4:** Why might an index on a column matching 90% of a large table not help?

**Answer:** Because the query needs most of the table. Using the index and fetching many rows may cost more than a sequential scan.

**Question 5:** Why can an index make inserts slower?

**Answer:** PostgreSQL has to update the table and maintain the relevant index entries.

## 14. Your Friday task

The README asks you to add an index, run `EXPLAIN` before and after, and explain the change.

A good explanation follows this pattern:

1. Before the index, the plan used ______.
2. I created an index on ______ using `CREATE INDEX`.
3. After the index, the plan used ______ (or PostgreSQL continued to use the previous plan).
4. The plan changed/did not change because ______.
5. The index is useful when ______.
6. A trade-off is ______.

Remember: your job is not to claim that every index makes every query faster. Your job is to read the evidence and explain the planner's choice.